# Exercise 3 — Connecting Python and SQL: An End-to-End ETL

**Time:** ~25 minutes

**Goal:** Build a full **ETL pipeline** in Python:

> `[ SQLite ] → [ Extract via SQL ] → [ Transform in pandas ] → [ Load to CSV ]`

This is the same ETL diagram from Session 1. The new part is that **you** are writing the pipeline code instead of clicking around in a BI tool.

**Prereq:** Run `python init_orders.py` once to create `cafe_data.db`.

## Why connect Python to SQL?

- SQL is **best at filtering and joining** at the database — close to the data, fast.
- pandas is **best at flexible transformations** — pivots, complex KPIs, what-ifs.
- Real analytics teams use **both**: SQL extracts a focused slice, pandas reshapes it.
- Python lets you **automate** the pipeline (run nightly, send a report, refresh a dashboard).

Schema in `cafe_data.db`:

```
 menu (id, name, price, category)
 customers (id, name, signup_date)
 orders (id, customer_id, menu_id, quantity, order_date)
```

## EXTRACT — pull joined data out of SQLite

In [ ]:
import sqlite3
import pandas as pd

conn = sqlite3.connect('cafe_data.db')

# Sanity check — what tables are in there?
pd.read_sql_query(
    "SELECT name FROM sqlite_master WHERE type='table'",
    conn,
)

**TODO:** Write a SQL query that **joins** orders with menu and customers, returning one row per order line with these columns:

`order_id, order_date, customer_name, product, category, quantity, price, revenue`

Note `revenue` is `quantity * price` — compute it in SQL.

*Recall from last session:* `INNER JOIN ... ON ...`

In [ ]:
query = """
    SELECT
        o.id              AS order_id,
        o.order_date      AS order_date,
        c.name            AS customer_name,
        m.name            AS product,
        m.category        AS category,
        o.quantity        AS quantity,
        m.price           AS price,
        o.quantity * m.price AS revenue
    FROM orders o
    INNER JOIN menu      m ON m.id = o.menu_id
    INNER JOIN customers c ON c.id = o.customer_id
    ORDER BY o.order_date, o.id
"""

df = pd.read_sql_query(query, conn)
df.head()

In [ ]:
print(f"Extracted {len(df)} order lines")
df.info()

## TRANSFORM — compute the three KPIs in pandas

Recall from Session 1: KPIs are **measurable, actionable, aligned with goals**. Café Cozy's owner wants to know:

1. **Top-selling product by revenue** — what should we never run out of?
2. **Revenue by category** — is the bakery worth the kitchen space?
3. **Average basket size per order** — are people upselling, or just grabbing one drink?

### KPI 1 — Top-selling product by revenue

**TODO:** Group by `product`, sum `revenue`, sort descending.

In [ ]:
top_products = (
    df.groupby('product')['revenue']
      .sum()
      .sort_values(ascending=False)
      .round(2)
)
top_products

### KPI 2 — Revenue and order count by category

**TODO:** Group by `category`. Use `.agg()` to compute **two** aggregations at once: total revenue and number of order lines.

In [ ]:
by_category = (
    df.groupby('category')
      .agg(
          revenue=('revenue', 'sum'),
          order_lines=('order_id', 'count'),
      )
      .round(2)
      .sort_values('revenue', ascending=False)
)
by_category

### KPI 3 — Average basket size per order

An "order" is one `order_id`. Basket size = total revenue for that order.

**TODO:** Group by `order_id`, sum revenue per order, then take the mean.

In [ ]:
basket_per_order = df.groupby('order_id')['revenue'].sum()
avg_basket = basket_per_order.mean().round(2)
median_basket = basket_per_order.median().round(2)

print(f"Average basket size: €{avg_basket}")
print(f"Median basket size:  €{median_basket}")
print(f"Total orders:        {len(basket_per_order)}")

## LOAD — save results so somebody else can use them

A KPI nobody can see is a KPI that does nothing. We'll write our results to CSV — that's the file format dashboards (Power BI, Looker, Data Studio) and spreadsheets all accept.

In [ ]:
# Save each KPI to its own CSV
top_products.to_csv('kpi_top_products.csv', header=['revenue_eur'])
by_category.to_csv('kpi_by_category.csv')

# Also save a one-line summary
summary = pd.DataFrame([{
    'metric': 'avg_basket_eur',     'value': avg_basket,
}, {
    'metric': 'median_basket_eur',  'value': median_basket,
}, {
    'metric': 'total_orders',       'value': len(basket_per_order),
}, {
    'metric': 'total_revenue_eur',  'value': df['revenue'].sum().round(2),
}])
summary.to_csv('kpi_summary.csv', index=False)

print('Wrote:')
for f in ['kpi_top_products.csv', 'kpi_by_category.csv', 'kpi_summary.csv']:
    print(f'  {f}')

In [ ]:
# Always close DB connections when done
conn.close()

## 🚀 Stretch goals — pick one

1. **Most loyal customer** — group by `customer_name`, count distinct order_ids, find the customer with the most orders.
2. **Daily revenue trend** — group by `order_date`, sum revenue, plot it with `.plot()`.
3. **Filter then aggregate** — what's the top-selling Bakery item? (use `df[df['category'] == 'Bakery']` first.)
4. **Write back to the DB** — save your KPI tables to a new SQLite table using `df.to_sql('kpi_top_products', conn, if_exists='replace')`.

In [ ]:
# YOUR STRETCH CODE HERE


### Where this fits — Data Science Lifecycle recap

Today's pipeline mapped to Salma's lifecycle slide:

| Phase | What we did |
|---|---|
| Business understanding | "Owner wants top seller, category revenue, basket size" |
| Data mining / extract | SQL JOIN across orders + menu + customers |
| Data cleaning | Already clean here — but Exercise 2 showed how |
| Data exploration | `.head()`, `.info()`, `.describe()` |
| Feature engineering | `revenue = quantity * price` |
| Predictive modeling | *(skipped — not all analyses need ML)* |
| Data visualization | The CSVs feed a dashboard — Power BI / Looker takes it from here |

**You just built a real data pipeline.** Every analytics team in industry runs scripts like this — often automated to run nightly. The shape doesn't change.